# config.py

In [ ]:
# config.py
import torch

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_MC = 3000
N_POINTS = 101
N_CHANNELS = 12
N_OUTPUTS = 11

N_VTH = 4        
N_CONDUCTANCE = 7      

N_INPUTS = N_POINTS * N_CHANNELS 

save_dir = "mosfet_mlp_3000_10_10_25C"

param_names = [
    "vth0", "k1", "k2", "nfactor", "u0", "ua", "ub", "a0", "ags", "uc", "keta"
]

channel_names = [
    "logId_Vds_0p1_Vbs_0", 
    "logId_Vds_0p1_Vbs_2", 
    "logId_Vds_0p1_Vbs_6",
    "logId_Vds_0p1_Vbs_10",

    "Id_Vds_0p1_Vbs_0", 
    "Id_Vds_0p1_Vbs_2", 
    "Id_Vds_0p1_Vbs_6",
    "Id_Vds_0p1_Vbs_10",
    "Id_Vds_5p0_Vbs_0", 
    "Id_Vds_5p0_Vbs_2",

    "dId_dVg_Vds_0p1_Vbs_0",

    "Vg"
]

conductance_feature_names = [
    "idlin_0vbs",
    "idlin_m2vbs",
    "idlin_m6vbs",
    "idlin_m10vbs",
    "idsat_0vbs",
    "idsat_m2vbs",
    "max_dId_dVg_0vbs",
]


CONDUCTANCE_FEATURE_CHANNELS = [4, 5, 6, 7, 8, 9, 10]

Y_LOG_COLS = [4, 5, 6, 9]  
Y_NEG_LOG_COLS = []    
EPS = 1e-30

# model_mlp.py

In [ ]:
# model_mlp.py
import os
import numpy as np
import torch
from torch import nn

class MosfetMLP(nn.Module):
    def __init__(self, n_inputs=N_INPUTS, n_outputs=N_OUTPUTS, use_bounds=True):
        super(MosfetMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(n_inputs, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(p=0.10),

            nn.Linear(2048, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(p=0.10),

            nn.Linear(2048, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(p=0.10),

            nn.Linear(2048, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(p=0.10),

            nn.Linear(2048, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(p=0.05),

            nn.Linear(2048, n_outputs)
        )

        # ---- bornes (ordre param_names, identique à scaler_y) ----
        self.use_bounds = use_bounds
        if use_bounds:
            lo = np.load(os.path.join(save_dir, "param_bounds_lower_norm.npy"))
            hi = np.load(os.path.join(save_dir, "param_bounds_upper_norm.npy"))
            self.register_buffer("lo", torch.tensor(lo, dtype=torch.float32))
            self.register_buffer("hi", torch.tensor(hi, dtype=torch.float32))

    def forward(self, x):
        raw = self.network(x)        # (B, 13) ordre param_names directement
        if self.use_bounds:
            # sigmoïde -> [0,1] puis remise à l'échelle [lo, hi]
            return self.lo + (self.hi - self.lo) * torch.sigmoid(raw)
        return raw

# data_prep.py

In [ ]:
# data_prep.py
import os, joblib, numpy as np, pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

os.makedirs(save_dir, exist_ok=True)
np.random.seed(SEED)

# ---- chargement X (.npz) ----
data = np.load("dataset_3000/mosfet_X_dataset_3000.npz")
X = data["X"].astype(np.float32)
print("X chargé, shape =", X.shape)

assert X.shape == (N_MC, N_POINTS, N_CHANNELS), \
    f"X devrait être {(N_MC, N_POINTS, N_CHANNELS)}, reçu {X.shape}"

# ---- chargement Y + transformations log ----
Y = pd.read_csv(
    "dataset_3000/Y_3000.csv",
    header=None,
    sep=";"
).values.astype(np.float32)
Yp = Y.copy()

Yp[:, Y_LOG_COLS] = np.log10(np.abs(Yp[:, Y_LOG_COLS]) + EPS)
Yp[:, Y_NEG_LOG_COLS] = np.log10(np.abs(Yp[:, Y_NEG_LOG_COLS]) + EPS)

# ---- chargement Vth pour loss uniquement ----
VTH = pd.read_csv(
    "dataset_3000/vth_extracted_3000.csv",
    header=None,
    sep=";"
).values.astype(np.float32)

print("VTH shape :", VTH.shape)
assert VTH.shape == (N_MC, N_VTH), f"Attendu ({N_MC},{N_VTH})"

# ---- chargement conductance features pour loss uniquement ----
CONDUCTANCE = pd.read_csv(
    "dataset_3000/currents_and_gmmax_extracted_3000.csv",
    header=None,
    sep=";"
).values.astype(np.float32)

print("CONDUCTANCE shape :", CONDUCTANCE.shape)
assert CONDUCTANCE.shape == (N_MC, N_CONDUCTANCE), f"Attendu ({N_MC},{N_CONDUCTANCE})"

# ---- split IDENTIQUE pour X, Y, VTH, CONDUCTANCE ----
(
    X_tr, X_tmp,
    Y_tr, Y_tmp,
    V_tr, V_tmp,
    C_tr, C_tmp
) = train_test_split(
    X, Yp, VTH, CONDUCTANCE,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

(
    X_val, X_te,
    Y_val, Y_te,
    V_val, V_te,
    C_val, C_te
) = train_test_split(
    X_tmp, Y_tmp, V_tmp, C_tmp,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# ---- scaler X par canal ----
scaler_x = MinMaxScaler((0, 1))

def norm_x(A, fit=False):
    A2 = A.reshape(-1, N_CHANNELS)
    A2 = scaler_x.fit_transform(A2) if fit else scaler_x.transform(A2)
    return A2.reshape(A.shape).astype(np.float32)

X_tr_n  = norm_x(X_tr, fit=True)
X_val_n = norm_x(X_val)
X_te_n  = norm_x(X_te)

# ---- normalisation conductance avec les mêmes min/max que les canaux X correspondants ----
conductance_channels = np.array(CONDUCTANCE_FEATURE_CHANNELS, dtype=int)

conductance_min = scaler_x.data_min_[conductance_channels].astype(np.float32)
conductance_max = scaler_x.data_max_[conductance_channels].astype(np.float32)

def norm_conductance(C):
    return ((C - conductance_min) / (conductance_max - conductance_min + 1e-12)).astype(np.float32)

C_tr_n  = norm_conductance(C_tr)
C_val_n = norm_conductance(C_val)
C_te_n  = norm_conductance(C_te)

print("CONDUCTANCE normalisés, shape train :", C_tr_n.shape)

# ---- scaler Y ----
scaler_y = MinMaxScaler((0, 1))

Y_tr_n  = scaler_y.fit_transform(Y_tr).astype(np.float32)
Y_val_n = scaler_y.transform(Y_val).astype(np.float32)
Y_te_n  = scaler_y.transform(Y_te).astype(np.float32)

# ---- format aplati canal-major : (N, 12, 101) -> (N, 1212) ----
def to_flat(A):
    return np.transpose(A, (0, 2, 1)).reshape(A.shape[0], -1).copy()

# ---- entrée MLP : uniquement les courbes aplaties ----
X_tr_flat  = to_flat(X_tr_n).astype(np.float32)
X_val_flat = to_flat(X_val_n).astype(np.float32)
X_te_flat  = to_flat(X_te_n).astype(np.float32)

print("Vecteur d'entrée train :", X_tr_flat.shape)
assert X_tr_flat.shape[1] == N_INPUTS, f"Entrée attendue {N_INPUTS}, reçu {X_tr_flat.shape[1]}"

# ---- Normaliser Vth dans l'espace de l'axe Vg pour la loss Vth ----
vg_min_sc = float(scaler_x.data_min_[11])
vg_max_sc = float(scaler_x.data_max_[11])

Vth_tr_n  = ((V_tr  - vg_min_sc) / (vg_max_sc - vg_min_sc + 1e-12)).astype(np.float32)
Vth_val_n = ((V_val - vg_min_sc) / (vg_max_sc - vg_min_sc + 1e-12)).astype(np.float32)

# ---- sauvegarde arrays ----
np.save(f"{save_dir}/X_train_flat.npy", X_tr_flat)
np.save(f"{save_dir}/X_val_flat.npy",   X_val_flat)
np.save(f"{save_dir}/X_test_flat.npy",  X_te_flat)

np.save(f"{save_dir}/Y_train_norm.npy", Y_tr_n)
np.save(f"{save_dir}/Y_val_norm.npy",   Y_val_n)
np.save(f"{save_dir}/Y_test_norm.npy",  Y_te_n)
np.save(f"{save_dir}/Y_test_processed.npy", Y_te)

# Loss Vth uniquement
np.save(f"{save_dir}/Vth_train_norm.npy", Vth_tr_n)
np.save(f"{save_dir}/Vth_val_norm.npy",   Vth_val_n)

# Loss conductance uniquement
np.save(f"{save_dir}/Conductance_train_norm.npy", C_tr_n)
np.save(f"{save_dir}/Conductance_val_norm.npy",   C_val_n)
np.save(f"{save_dir}/Conductance_test_norm.npy",  C_te_n)

# Loss Vth
np.save(f"{save_dir}/Vth_train_norm.npy", Vth_tr_n)
np.save(f"{save_dir}/Vth_val_norm.npy",   Vth_val_n)

# Test : versions normalisée et brute pour l'évaluation
Vth_te_n = ((V_te - vg_min_sc) / (vg_max_sc - vg_min_sc + 1e-12)).astype(np.float32)

np.save(f"{save_dir}/Vth_test_norm.npy", Vth_te_n)
np.save(f"{save_dir}/Vth_test_raw.npy",  V_te.astype(np.float32))

print("Vth loss norm shape train :", Vth_tr_n.shape)
print("Conductance loss norm shape train :", C_tr_n.shape)

# ---- BORNES paramètres SPICE ----
lower = np.array([0.55, 0.60, 0.03, 0.60, -1.40, -9.35, 
                  -19.00, 0.40, -0.10, -11.70, -0.02],
                 dtype=np.float32)
upper = np.array([0.75, 0.80, 0.08, 1.40, -1.20, -8.95, 
                  -17.80, 1.20, 0.10, -10.30, 0.02],
                 dtype=np.float32)

lo_n = scaler_y.transform(lower.reshape(1, -1)).astype(np.float32)[0]
hi_n = scaler_y.transform(upper.reshape(1, -1)).astype(np.float32)[0]

lower_norm = np.minimum(lo_n, hi_n)
upper_norm = np.maximum(lo_n, hi_n)

np.save(f"{save_dir}/param_bounds_lower_norm.npy", lower_norm)
np.save(f"{save_dir}/param_bounds_upper_norm.npy", upper_norm)

print("Bornes normalisées :", lower_norm, upper_norm)

# ---- sauvegarde scalers ----
joblib.dump(scaler_x, f"{save_dir}/scaler_x_minmax.pkl")
joblib.dump(scaler_y, f"{save_dir}/scaler_y_minmax.pkl")

print("Préparation terminée.")
print("X_train_flat :", X_tr_flat.shape)
print("Y_train_norm :", Y_tr_n.shape)

# surrogate_vth_loss.py 

In [ ]:
# ============================================================
# surrogate_vth_loss.py  ← NOUVELLE cellule (après config.py)
# ============================================================
import os, joblib, numpy as np, torch
from torch import nn

# ---- Redéfinir MosfetSurrogate (doit correspondre à surrogate.ipynb) ----
N_CURVE = N_CHANNELS * N_POINTS          # 12 × 101 = 1212

class MosfetSurrogate(nn.Module):
    def __init__(self, n_params=N_OUTPUTS, n_curve=N_CURVE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_params, 512),  nn.BatchNorm1d(512),  nn.ReLU(),
            nn.Linear(512,  1024),     nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
            nn.Linear(2048, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
            nn.Linear(2048, n_curve),
        )
    def forward(self, p):
        return self.net(p)

# ---- Hyperparamètre : poids de la loss Vth ----
LAMBDA_VTH = 0.285      

# ---- Charger le surrogate et le GELER ----
SURROGATE_DIR  = "mosfet_surrogate_3000_10_10_25C"
SURROGATE_PATH = os.path.join(SURROGATE_DIR, "surrogate_best.pth")

surrogate = MosfetSurrogate().to(device)
surrogate.load_state_dict(torch.load(SURROGATE_PATH, map_location=device))
surrogate.eval()
for p in surrogate.parameters():
    p.requires_grad_(False)          
print(f"Surrogate gelé chargé depuis {SURROGATE_PATH}")

# ---- Axe Vg normalisé (constant : même sweep pour tous les échantillons) ----
scaler_x_obj = joblib.load(os.path.join(save_dir, "scaler_x_minmax.pkl"))
vg_min_sc  = float(scaler_x_obj.data_min_[11])
vg_max_sc  = float(scaler_x_obj.data_max_[11])
vg_raw     = np.linspace(0.0, 5.0, N_POINTS)       
vg_norm_np = (vg_raw - vg_min_sc) / (vg_max_sc - vg_min_sc + 1e-12)
vg_norm_t  = torch.tensor(vg_norm_np, dtype=torch.float32, device=device)

# ---- Cible logId = -7 en espace normalisé, une valeur par canal Vbs ----
logid_min = scaler_x_obj.data_min_[:4].astype(np.float32)
logid_max = scaler_x_obj.data_max_[:4].astype(np.float32)
logid_tgt_np = (-7.0 - logid_min) / (logid_max - logid_min + 1e-12)
logid_tgt_t  = torch.tensor(logid_tgt_np, dtype=torch.float32, device=device)
print("Cible logId=-7 normalisée :", logid_tgt_np.round(4))


# ---------------------------------------------------------------
def interp_at_vth(logid_curve, vg_axis, vth_norm):
    B = logid_curve.shape[0]
    N = vg_axis.shape[0]                          

    # Indice de l'intervalle (non-différentiable : ok, gradient via y0/y1)
    idx = torch.searchsorted(
        vg_axis.contiguous(), vth_norm.contiguous()
    ).clamp(1, N - 1)                             

    x0 = vg_axis[idx - 1]                       
    x1 = vg_axis[idx]                           
    y0 = logid_curve[torch.arange(B, device=logid_curve.device), idx - 1]  
    y1 = logid_curve[torch.arange(B, device=logid_curve.device), idx]     

    # Coefficient d'interpolation (constant vis-à-vis des paramètres)
    t = ((vth_norm - x0) / (x1 - x0 + 1e-8)).clamp(0.0, 1.0) 
    return y0 + t * (y1 - y0)               


def compute_vth_loss(x_flat_pred, vth_norm_batch):
    total = torch.tensor(0.0, device=x_flat_pred.device)
    for j in range(4):
        # Canal j de logId : indices [j*101 : (j+1)*101] dans le vecteur aplati
        logid_j = x_flat_pred[:, j * N_POINTS : (j + 1) * N_POINTS]  
        logid_hat = interp_at_vth(logid_j, vg_norm_t, vth_norm_batch[:, j])  
        total = total + ((logid_hat - logid_tgt_t[j]) ** 2).mean()
    return total / 4.0

# ---- Hyperparamètre : poids de la loss extra features ----
LAMBDA_CONDUCTANCE = 0.285  
def compute_conductance_loss(x_flat_pred, conductance_norm_batch):
    vals = []

    # Vg = 5 V correspond au dernier point du sweep : index 100
    idx_vg5 = N_POINTS - 1

    # idlin canaux 4..7 à Vg=5 V
    for ch in [4, 5, 6, 7]: 
        vals.append(x_flat_pred[:, ch * N_POINTS + idx_vg5])

    # idsat canaux 8..9 à Vg=5 V
    for ch in [8, 9]:
        vals.append(x_flat_pred[:, ch * N_POINTS + idx_vg5])

    # gmmax : max canal 10
    gm_curve = x_flat_pred[:, 10 * N_POINTS : 11 * N_POINTS]  # (B, 101)
    gmmax = gm_curve.max(dim=1).values
    vals.append(gmmax)

    conductance_hat = torch.stack(vals, dim=1)  # (B, 7)

    return ((conductance_hat - conductance_norm_batch) ** 2).mean()

# train.py

In [ ]:
# train.py
import os, numpy as np, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device, "| CUDA:", torch.cuda.is_available())

batch_size, num_epochs, patience, lr = 128, 200, 40, 3e-5
best_model_path = os.path.join(save_dir, "mlp_best.pth")

# ---- Charger données + Vth + CONDUCTANCE ----
X_tr  = torch.tensor(np.load(f"{save_dir}/X_train_flat.npy"), dtype=torch.float32)
X_val = torch.tensor(np.load(f"{save_dir}/X_val_flat.npy"),   dtype=torch.float32)

Y_tr  = torch.tensor(np.load(f"{save_dir}/Y_train_norm.npy"), dtype=torch.float32)
Y_val = torch.tensor(np.load(f"{save_dir}/Y_val_norm.npy"),   dtype=torch.float32)

VTH_TR  = torch.tensor(np.load(f"{save_dir}/Vth_train_norm.npy"), dtype=torch.float32)
VTH_VAL = torch.tensor(np.load(f"{save_dir}/Vth_val_norm.npy"),   dtype=torch.float32)

CONDUCTANCE_TR  = torch.tensor(np.load(f"{save_dir}/Conductance_train_norm.npy"), dtype=torch.float32)
CONDUCTANCE_VAL = torch.tensor(np.load(f"{save_dir}/Conductance_val_norm.npy"),   dtype=torch.float32)

print(
    "X_flat:", X_tr.shape,
    "Y:", Y_tr.shape,
    "VTH:", VTH_TR.shape,
    "CONDUCTANCE:", CONDUCTANCE_TR.shape
)

train_loader = DataLoader(
    TensorDataset(X_tr, Y_tr, VTH_TR, CONDUCTANCE_TR),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    TensorDataset(X_val, Y_val, VTH_VAL, CONDUCTANCE_VAL),
    batch_size=batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

# ---- Modèle MLP ----
model = MosfetMLP().to(device)
print("Params entraînables MLP :", sum(p.numel() for p in model.parameters() if p.requires_grad))

# ---- Loss paramètres + optimizer ----
criterion = nn.HuberLoss(delta=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    "min",
    factor=0.5,
    patience=10,
    min_lr=1e-7
)

param_weights = torch.ones(N_OUTPUTS, device=device)
for hard in ["ub", "uc", "ags", "keta"]:
    param_weights[param_names.index(hard)] = 1.0

def weighted_loss(pred, target):
    return criterion(pred * param_weights, target * param_weights)

# ---- Boucle d'entraînement ----
train_losses, val_losses = [], []
best_val, patience_counter = np.inf, 0

for epoch in range(num_epochs):

    # ── TRAIN ─────────────────────────────────────────────
    model.train()

    run_total = run_param = run_vth = run_conductance = 0.0
    n = 0

    for Xb, Yb, Vb, Cb in train_loader:
        Xb = Xb.to(device, non_blocking=True)
        Yb = Yb.to(device, non_blocking=True)
        Vb = Vb.to(device, non_blocking=True)
        Cb = Cb.to(device, non_blocking=True)

        optimizer.zero_grad()

        # 1. Prédiction paramètres SPICE
        params_pred = model(Xb)

        # 2. Loss paramètres
        loss_p = weighted_loss(params_pred, Yb)

        # 3. Reconstruction courbes via surrogate gelé
        x_flat_pred = surrogate(params_pred)

        # 4. Loss physique Vth
        loss_v = compute_vth_loss(x_flat_pred, Vb)

        # 5. Loss physique conductance
        loss_c = compute_conductance_loss(x_flat_pred, Cb)

        # 6. Loss totale
        loss = loss_p + LAMBDA_VTH * loss_v + LAMBDA_CONDUCTANCE * loss_c

        loss.backward()
        optimizer.step()

        bs = Xb.size(0)

        run_total       += loss.item()   * bs
        run_param       += loss_p.item() * bs
        run_vth         += loss_v.item() * bs
        run_conductance += loss_c.item() * bs

        n += bs

    # ── VALIDATION ───────────────────────────────────────
    model.eval()

    run_val_tot = run_val_p = run_val_v = run_val_c = 0.0
    nv = 0

    with torch.no_grad():
        for Xb, Yb, Vb, Cb in val_loader:
            Xb = Xb.to(device, non_blocking=True)
            Yb = Yb.to(device, non_blocking=True)
            Vb = Vb.to(device, non_blocking=True)
            Cb = Cb.to(device, non_blocking=True)

            params_pred = model(Xb)

            lp = weighted_loss(params_pred, Yb)

            xp = surrogate(params_pred)

            lv = compute_vth_loss(xp, Vb)
            lc = compute_conductance_loss(xp, Cb)

            val_loss = lp + LAMBDA_VTH * lv + LAMBDA_CONDUCTANCE * lc

            bs = Xb.size(0)

            run_val_tot += val_loss.item() * bs
            run_val_p   += lp.item()       * bs
            run_val_v   += lv.item()       * bs
            run_val_c   += lc.item()       * bs

            nv += bs

    t_tot = run_total / n
    t_p   = run_param / n
    t_v   = run_vth / n
    t_c   = run_conductance / n

    v_tot = run_val_tot / nv
    v_p   = run_val_p / nv
    v_v   = run_val_v / nv
    v_c   = run_val_c / nv

    scheduler.step(v_tot)

    train_losses.append(t_tot)
    val_losses.append(v_tot)

    print(
        f"Epoch [{epoch+1:03d}/{num_epochs}] "
        f"LR {optimizer.param_groups[0]['lr']:.2e} "
        f"Train {t_tot:.5f} "
        f"(param {t_p:.5f} + λvth {LAMBDA_VTH*t_v:.5f} + λconductance {LAMBDA_CONDUCTANCE*t_c:.5f}) "
        f"Val {v_tot:.5f} "
        f"(param {v_p:.5f} + λvth {LAMBDA_VTH*v_v:.5f} + λconductance {LAMBDA_CONDUCTANCE*v_c:.5f})"
    )

    if v_tot < best_val:
        best_val, patience_counter = v_tot, 0

        tmp_path = best_model_path + ".tmp"
        torch.save(model.state_dict(), tmp_path)

        if os.path.exists(best_model_path):
            os.remove(best_model_path)

        os.rename(tmp_path, best_model_path)

        print("  ✓ Meilleur modèle sauvegardé.")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping.")
        break

print("Meilleure val loss:", best_val)

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss totale")
plt.grid(True)
plt.legend()
plt.title(
    f"MLP + Vth loss + Conductance loss "
    f"(λvth={LAMBDA_VTH}, λconductance={LAMBDA_CONDUCTANCE})"
)
plt.show()

# predict.py

In [ ]:
# predict.py
import os
import joblib
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import TensorDataset, DataLoader


batch_size = 64

best_model_path = os.path.join(save_dir, "mlp_best.pth")

scaler_x = joblib.load(os.path.join(save_dir, "scaler_x_minmax.pkl"))
scaler_y = joblib.load(os.path.join(save_dir, "scaler_y_minmax.pkl"))


# ------------------------------------------------------------
# 1. Chargement données test
# ------------------------------------------------------------
X_test = np.load(os.path.join(save_dir, "X_test_flat.npy"))
Y_test_norm = np.load(os.path.join(save_dir, "Y_test_norm.npy"))

print("X_test:", X_test.shape, "Y_test_norm:", Y_test_norm.shape)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test_norm, dtype=torch.float32)

test_loader = DataLoader(
    TensorDataset(X_test_tensor, Y_test_tensor),
    batch_size=batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)


# ------------------------------------------------------------
# 2. Chargement modèle MLP
# ------------------------------------------------------------
model = MosfetMLP().to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("MLP loaded from:", best_model_path)


# ------------------------------------------------------------
# 3. Vérifier / charger le surrogate
# ------------------------------------------------------------
try:
    surrogate
    print("Surrogate déjà présent en mémoire.")
except NameError:
    print("Surrogate non trouvé en mémoire : rechargement...")

    SURROGATE_DIR = "mosfet_surrogate_3000_10_10_25C"

    SURROGATE_PATH = os.path.join(SURROGATE_DIR, "surrogate_best_first.pth")

    if not os.path.exists(SURROGATE_PATH):
        SURROGATE_PATH = os.path.join(SURROGATE_DIR, "surrogate_best.pth")

    N_CURVE = N_CHANNELS * N_POINTS

    class MosfetSurrogate(nn.Module):
        def __init__(self, n_params=N_OUTPUTS, n_curve=N_CURVE):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_params, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),

                nn.Linear(512, 1024),
                nn.BatchNorm1d(1024),
                nn.ReLU(),

                nn.Linear(1024, 2048),
                nn.BatchNorm1d(2048),
                nn.ReLU(),

                nn.Linear(2048, 2048),
                nn.BatchNorm1d(2048),
                nn.ReLU(),

                nn.Linear(2048, n_curve),
            )

        def forward(self, p):
            return self.net(p)

    surrogate = MosfetSurrogate().to(device)
    surrogate.load_state_dict(torch.load(SURROGATE_PATH, map_location=device))
    surrogate.eval()

    for p in surrogate.parameters():
        p.requires_grad_(False)

    print("Surrogate loaded from:", SURROGATE_PATH)

surrogate.eval()


# ------------------------------------------------------------
# 4. Inférence MLP + reconstruction surrogate
# ------------------------------------------------------------
all_pred_norm = []
all_curve_pred_flat_norm = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)

        # Paramètres SPICE prédits normalisés
        params_pred_norm = model(X_batch)

        # Courbes reconstruites par surrogate depuis les paramètres prédits
        curve_pred_flat_norm = surrogate(params_pred_norm)

        all_pred_norm.append(params_pred_norm.cpu().numpy())
        all_curve_pred_flat_norm.append(curve_pred_flat_norm.cpu().numpy())

Y_pred_norm = np.vstack(all_pred_norm)
X_curve_pred_flat_norm = np.vstack(all_curve_pred_flat_norm)

print("Y_pred_norm:", Y_pred_norm.shape)
print("X_curve_pred_flat_norm:", X_curve_pred_flat_norm.shape)


# ------------------------------------------------------------
# 5. Comparaison paramètres SPICE
# ------------------------------------------------------------
Y_pred = scaler_y.inverse_transform(Y_pred_norm).copy()
Y_true = scaler_y.inverse_transform(Y_test_norm).copy()

# Inversion des transformations log appliquées dans data_prep.py
Y_pred[:, Y_LOG_COLS] = 10 ** Y_pred[:, Y_LOG_COLS]
Y_true[:, Y_LOG_COLS] = 10 ** Y_true[:, Y_LOG_COLS]

Y_pred[:, Y_NEG_LOG_COLS] = -10 ** Y_pred[:, Y_NEG_LOG_COLS]
Y_true[:, Y_NEG_LOG_COLS] = -10 ** Y_true[:, Y_NEG_LOG_COLS]

mae_per_param = np.mean(np.abs(Y_pred - Y_true), axis=0)
rmse_per_param = np.sqrt(np.mean((Y_pred - Y_true) ** 2, axis=0))

denom = np.abs(Y_true) + 1e-30

idx_k2 = param_names.index("k2")
idx_keta = param_names.index("keta")

# Dénominateurs spécifiques déjà utilisés dans ta cellule d'origine
denom[:, idx_k2] = 0.08
denom[:, idx_keta] = 0.2

rel_err = np.mean(np.abs(Y_pred - Y_true) / denom, axis=0)

print("\n" + "=" * 90)
print("ERREURS PARAMÈTRES SPICE")
print("=" * 90)

for name, mae, rmse, rel in zip(param_names, mae_per_param, rmse_per_param, rel_err):
    print(f"{name:>10s} | MAE = {mae:.6e} | RMSE = {rmse:.6e} | Rel = {100 * rel:.3f} %")


# ------------------------------------------------------------
# Fonction générique : graphes prédiction vs vrai groupés
# ------------------------------------------------------------
def plot_pred_vs_true_grouped(
    title,
    names,
    pred,
    true,
    ncols=3,
    figsize_per_subplot=(5, 5),
    save_path=None
):

    n_features = len(names)
    nrows = int(np.ceil(n_features / ncols))

    fig_width = figsize_per_subplot[0] * ncols
    fig_height = figsize_per_subplot[1] * nrows

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(fig_width, fig_height)
    )

    axes = np.array(axes).reshape(-1)

    for j, name in enumerate(names):
        ax = axes[j]

        p = pred[:, j]
        t = true[:, j]

        valid = np.isfinite(p) & np.isfinite(t)

        if valid.sum() == 0:
            ax.set_title(f"{name} — aucune valeur valide")
            ax.axis("off")
            continue

        p_valid = p[valid]
        t_valid = t[valid]

        ax.scatter(t_valid, p_valid, s=8, alpha=0.4)

        mn = min(t_valid.min(), p_valid.min())
        mx = max(t_valid.max(), p_valid.max())

        ax.plot([mn, mx], [mn, mx], "r--", linewidth=1.5)

        ax.set_xlabel(f"Vrai {name}")
        ax.set_ylabel(f"Prédit {name}")
        ax.set_title(name)
        ax.grid(True)

    # Cacher les subplots inutilisés
    for k in range(n_features, len(axes)):
        axes[k].axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path is not None:
        plt.savefig(save_path, dpi=300)
        print(f"Figure sauvegardée : {save_path}")

    plt.show()


# ------------------------------------------------------------
# Figure groupée : tous les paramètres SPICE
# ------------------------------------------------------------
plot_pred_vs_true_grouped(
    title="Prédiction vs vrai — Paramètres SPICE",
    names=param_names,
    pred=Y_pred,
    true=Y_true,
    ncols=4,
    figsize_per_subplot=(5, 5),
    save_path=os.path.join(save_dir, "grouped_parameters_pred_vs_true.png")
)


# ------------------------------------------------------------
# 6. Fonctions utilitaires pour reconstruire les courbes brutes
# ------------------------------------------------------------
def flat_norm_to_raw_curves_batch(x_flat_norm):
    n = x_flat_norm.shape[0]

    x_norm = x_flat_norm.reshape(n, N_CHANNELS, N_POINTS)
    x_norm = np.transpose(x_norm, (0, 2, 1))  # (N, 101, 12)

    x_raw = np.empty_like(x_norm, dtype=np.float32)

    for i in range(n):
        x_raw[i] = scaler_x.inverse_transform(x_norm[i]).astype(np.float32)

    return x_raw


def extract_vth_from_curve(vg, logid, target=-7.0):
    y = logid - target

    # Si un point tombe exactement sur la cible
    idx_exact = np.where(np.isclose(y, 0.0, atol=1e-12))[0]
    if len(idx_exact) > 0:
        return float(vg[idx_exact[0]])

    # Recherche du changement de signe
    sign_change = np.where(y[:-1] * y[1:] <= 0.0)[0]

    if len(sign_change) == 0:
        return np.nan

    k = sign_change[0]

    x0, x1 = vg[k], vg[k + 1]
    y0, y1 = logid[k], logid[k + 1]

    # Interpolation linéaire : target entre y0 et y1
    if abs(y1 - y0) < 1e-30:
        return float(x0)

    return float(x0 + (target - y0) * (x1 - x0) / (y1 - y0))


def extract_vth_batch_from_raw_curves(X_raw, target=-7.0):
    n = X_raw.shape[0]
    Vth = np.full((n, 4), np.nan, dtype=np.float32)

    for i in range(n):
        vg = X_raw[i, :, 11]

        for j in range(4):
            logid = X_raw[i, :, j]
            Vth[i, j] = extract_vth_from_curve(vg, logid, target=target)

    return Vth


def extract_conductance_features_from_raw_curves(X_raw):
    n = X_raw.shape[0]
    F = np.zeros((n, 7), dtype=np.float32)

    idx_vg5 = N_POINTS - 1

    # Idlin : canaux 4..7 à Vg=5 V
    F[:, 0] = X_raw[:, idx_vg5, 4]
    F[:, 1] = X_raw[:, idx_vg5, 5]
    F[:, 2] = X_raw[:, idx_vg5, 6]
    F[:, 3] = X_raw[:, idx_vg5, 7]

    # Idsat : canaux 8..9 à Vg=5 V
    F[:, 4] = X_raw[:, idx_vg5, 8]
    F[:, 5] = X_raw[:, idx_vg5, 9]

    # Gmmax : max canal 10
    F[:, 6] = np.max(X_raw[:, :, 10], axis=1)

    return F


def print_feature_metrics(title, names, pred, true, custom_denom=None):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    for j, name in enumerate(names):
        p = pred[:, j]
        t = true[:, j]

        valid = np.isfinite(p) & np.isfinite(t)

        if valid.sum() == 0:
            print(f"{name:>18s} | aucune valeur valide")
            continue

        p = p[valid]
        t = t[valid]

        mae = np.mean(np.abs(p - t))
        rmse = np.sqrt(np.mean((p - t) ** 2))

        if custom_denom is not None and custom_denom[j] is not None:
            denom = custom_denom[j]
            rel = np.mean(np.abs(p - t) / denom)
        else:
            denom = np.abs(t) + 1e-30
            rel = np.mean(np.abs(p - t) / denom)

        print(
            f"{name:>18s} | "
            f"MAE = {mae:.6e} | "
            f"RMSE = {rmse:.6e} | "
            f"Rel = {100 * rel:.3f} %"
        )


# ------------------------------------------------------------
# 7. Dénormalisation des courbes reconstruites par surrogate
# ------------------------------------------------------------
X_pred_raw = flat_norm_to_raw_curves_batch(X_curve_pred_flat_norm)

print("\nX_pred_raw:", X_pred_raw.shape)


# ------------------------------------------------------------
# 8. Comparaison Vth
# ------------------------------------------------------------
vth_test_raw_path = os.path.join(save_dir, "Vth_test_raw.npy")

if not os.path.exists(vth_test_raw_path):
    raise FileNotFoundError(
        "Le fichier Vth_test_raw.npy est introuvable. "
        "Relance d'abord data_prep.py après avoir ajouté la sauvegarde de V_te."
    )

Vth_true = np.load(vth_test_raw_path).astype(np.float32)

# Vth prédit extrait des courbes reconstruites surrogate(MLP params)
Vth_pred = extract_vth_batch_from_raw_curves(X_pred_raw, target=-7.0)

vth_names = [
    "vth_0vbs",
    "vth_m2vbs",
    "vth_m6vbs",
    "vth_m10vbs",
]

print_feature_metrics(
    title="ERREURS FEATURES VTH : Vth extrait depuis surrogate(MLP params)",
    names=vth_names,
    pred=Vth_pred,
    true=Vth_true
)

# Optionnel : Vth aussi groupé dans une figure
plot_pred_vs_true_grouped(
    title="Prédiction vs vrai — Vth",
    names=vth_names,
    pred=Vth_pred,
    true=Vth_true,
    ncols=2,
    figsize_per_subplot=(5, 5),
    save_path=os.path.join(save_dir, "grouped_vth_pred_vs_true.png")
)


# ------------------------------------------------------------
# 9. Comparaison Idlin / Idsat / Gmmax
# ------------------------------------------------------------
conductance_test_raw_path = os.path.join(save_dir, "Conductance_test_raw.npy")

if os.path.exists(conductance_test_raw_path):
    Conductance_true = np.load(conductance_test_raw_path).astype(np.float32)
else:
    print(
        "\nConductance_test_raw.npy introuvable. "
        "Utilisation de Conductance_test_norm.npy puis dénormalisation via scaler_x."
    )

    Conductance_test_norm = np.load(
        os.path.join(save_dir, "Conductance_test_norm.npy")
    ).astype(np.float32)

    conductance_channels = np.array(CONDUCTANCE_FEATURE_CHANNELS, dtype=int)

    conductance_min = scaler_x.data_min_[conductance_channels].astype(np.float32)
    conductance_max = scaler_x.data_max_[conductance_channels].astype(np.float32)

    Conductance_true = (
        Conductance_test_norm * (conductance_max - conductance_min + 1e-12)
        + conductance_min
    ).astype(np.float32)

# Features prédites extraites des courbes reconstruites
Conductance_pred = extract_conductance_features_from_raw_curves(X_pred_raw)

conductance_names = [
    "idlin_0vbs",
    "idlin_m2vbs",
    "idlin_m6vbs",
    "idlin_m10vbs",
    "idsat_0vbs",
    "idsat_m2vbs",
    "gmmax_0vbs",
]

print_feature_metrics(
    title="ERREURS FEATURES IDLIN / IDSAT / GMMAX : extraites depuis surrogate(MLP params)",
    names=conductance_names,
    pred=Conductance_pred,
    true=Conductance_true
)

# ------------------------------------------------------------
# Figure groupée : Idlin / Idsat / Gmmax
# ------------------------------------------------------------
plot_pred_vs_true_grouped(
    title="Prédiction vs vrai — Idlin / Idsat / Gmmax",
    names=conductance_names,
    pred=Conductance_pred,
    true=Conductance_true,
    ncols=3,
    figsize_per_subplot=(5, 5),
    save_path=os.path.join(save_dir, "grouped_currents_pred_vs_true.png")
)


# ------------------------------------------------------------
# 10. Sauvegarde des résultats
# ------------------------------------------------------------
np.save(os.path.join(save_dir, "Y_pred_test_mlp.npy"), Y_pred)
np.save(os.path.join(save_dir, "Y_true_test_mlp.npy"), Y_true)

np.save(os.path.join(save_dir, "Vth_pred_test_mlp.npy"), Vth_pred)
np.save(os.path.join(save_dir, "Vth_true_test_mlp.npy"), Vth_true)

np.save(os.path.join(save_dir, "Conductance_pred_test_mlp.npy"), Conductance_pred)
np.save(os.path.join(save_dir, "Conductance_true_test_mlp.npy"), Conductance_true)

print("\nPredictions saved:")
print(" - Y_pred_test_mlp.npy")
print(" - Y_true_test_mlp.npy")
print(" - Vth_pred_test_mlp.npy")
print(" - Vth_true_test_mlp.npy")
print(" - Conductance_pred_test_mlp.npy")
print(" - Conductance_true_test_mlp.npy")

print("\nFigures sauvegardées:")
print(" - grouped_parameters_pred_vs_true.png")
print(" - grouped_vth_pred_vs_true.png")
print(" - grouped_currents_pred_vs_true.png")

# predict_one_Xtest_sample.py

In [ ]:
# ============================================================
# predict_one_Xtest_sample.py
# Tester le MLP sur un échantillon de X_test + reconstruction surrogate
# ============================================================

import os, joblib, numpy as np, torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Choisir l'index de l'échantillon X_test à tester
# ------------------------------------------------------------
sample_index = 100 

best_model_path = os.path.join(save_dir, "mlp_best.pth")

# ------------------------------------------------------------
# 2. Charger scalers et données test
# ------------------------------------------------------------
scaler_x = joblib.load(os.path.join(save_dir, "scaler_x_minmax.pkl"))
scaler_y = joblib.load(os.path.join(save_dir, "scaler_y_minmax.pkl"))

X_test_flat = np.load(os.path.join(save_dir, "X_test_flat.npy"))       
Y_test_norm = np.load(os.path.join(save_dir, "Y_test_norm.npy"))       

print("X_test_flat shape :", X_test_flat.shape)
print("Y_test_norm shape :", Y_test_norm.shape)

if sample_index < 0 or sample_index >= X_test_flat.shape[0]:
    raise ValueError(f"sample_index doit être entre 0 et {X_test_flat.shape[0]-1}")

# ------------------------------------------------------------
# 3. Extraire l'échantillon choisi
# ------------------------------------------------------------
x_sample_full_norm = X_test_flat[sample_index:sample_index+1].astype(np.float32)  
y_true_norm = Y_test_norm[sample_index:sample_index+1].astype(np.float32)        

# Partie courbes normalisées uniquement : les 1212 premiers éléments
x_curve_true_flat_norm = x_sample_full_norm[:, :N_CHANNELS * N_POINTS]            

print("\nSample index :", sample_index)
print("x_sample_full_norm shape :", x_sample_full_norm.shape)
print("x_curve_true_flat_norm shape :", x_curve_true_flat_norm.shape)
print("y_true_norm shape :", y_true_norm.shape)

# ------------------------------------------------------------
# 4. Charger le modèle MLP
# ------------------------------------------------------------
model = MosfetMLP().to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("\nMLP loaded from :", best_model_path)

# ------------------------------------------------------------
# 5. Vérifier / charger le surrogate
# ------------------------------------------------------------

try:
    surrogate
    print("Surrogate déjà présent en mémoire.")
except NameError:
    print("Surrogate non trouvé en mémoire : rechargement...")

    SURROGATE_DIR  = "mosfet_surrogate_3000_10_10_25C"
    SURROGATE_PATH = os.path.join(SURROGATE_DIR, "surrogate_best_first.pth")

    N_CURVE = N_CHANNELS * N_POINTS

    class MosfetSurrogate(nn.Module):
        def __init__(self, n_params=N_OUTPUTS, n_curve=N_CURVE):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_params, 512),  nn.BatchNorm1d(512),  nn.ReLU(),
                nn.Linear(512,  1024),     nn.BatchNorm1d(1024), nn.ReLU(),
                nn.Linear(1024, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
                nn.Linear(2048, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
                nn.Linear(2048, n_curve),
            )

        def forward(self, p):
            return self.net(p)

    surrogate = MosfetSurrogate().to(device)
    surrogate.load_state_dict(torch.load(SURROGATE_PATH, map_location=device))
    surrogate.eval()

    for p in surrogate.parameters():
        p.requires_grad_(False)

    print("Surrogate loaded from :", SURROGATE_PATH)

surrogate.eval()

# ------------------------------------------------------------
# 6. Inférence MLP : paramètres SPICE prédits normalisés
# ------------------------------------------------------------
x_tensor = torch.tensor(x_sample_full_norm, dtype=torch.float32).to(device)

with torch.no_grad():
    y_pred_norm_t = model(x_tensor)                      
    x_curve_pred_flat_norm_t = surrogate(y_pred_norm_t)   

y_pred_norm = y_pred_norm_t.cpu().numpy()
x_curve_pred_flat_norm = x_curve_pred_flat_norm_t.cpu().numpy()

print("\ny_pred_norm shape :", y_pred_norm.shape)
print("x_curve_pred_flat_norm shape :", x_curve_pred_flat_norm.shape)

# ------------------------------------------------------------
# 7. Dénormaliser Y_pred et Y_true + inversion log
# ------------------------------------------------------------
Y_pred = scaler_y.inverse_transform(y_pred_norm).copy()
Y_true = scaler_y.inverse_transform(y_true_norm).copy()

# Inversion des transformations log appliquées pendant data_prep.py
Y_pred[:, Y_LOG_COLS] = 10 ** Y_pred[:, Y_LOG_COLS]
Y_true[:, Y_LOG_COLS] = 10 ** Y_true[:, Y_LOG_COLS]

Y_pred[:, Y_NEG_LOG_COLS] = -10 ** (Y_pred[:, Y_NEG_LOG_COLS])
Y_true[:, Y_NEG_LOG_COLS] = -10 ** (Y_true[:, Y_NEG_LOG_COLS])

# ------------------------------------------------------------
# 8. Afficher les paramètres SPICE prédits vs vrais
# ------------------------------------------------------------
print("\nParamètres SPICE : prédits vs vrais")
print("-" * 85)
print(f"{'param':>10s} | {'pred':>16s} | {'true':>16s} | {'abs err':>16s} | {'rel err %':>12s}")
print("-" * 85)

for j, name in enumerate(param_names):
    pred = float(Y_pred[0, j])
    true = float(Y_true[0, j])
    abs_err = abs(pred - true)

    denom = abs(true) + 1e-30
    if name == "k2":
        denom = 0.08
    elif name == "keta":
        denom = 0.2

    rel_err = 100.0 * abs_err / denom

    print(
        f"{name:>10s} | "
        f"{pred:16.8e} | "
        f"{true:16.8e} | "
        f"{abs_err:16.8e} | "
        f"{rel_err:12.4f}"
    )

# ------------------------------------------------------------
# 9. Dénormaliser les courbes X_true et X_pred
# ------------------------------------------------------------
def flat_norm_to_raw_curve(x_flat_norm):
    """
    Convertit un vecteur aplati normalisé canal-major (1, 1212)
    vers une matrice brute (101, 12), en inversant scaler_x.

    Format aplati utilisé dans data_prep.py :
        (N, 12, 101) -> (N, 1212)
    Donc pour un sample :
        (1212,) -> (12, 101) -> transpose -> (101, 12)
    """
    x_norm_101_12 = x_flat_norm.reshape(N_CHANNELS, N_POINTS).T 
    x_raw_101_12 = scaler_x.inverse_transform(x_norm_101_12)     
    return x_raw_101_12.astype(np.float32)

X_true_raw = flat_norm_to_raw_curve(x_curve_true_flat_norm[0])
X_pred_raw = flat_norm_to_raw_curve(x_curve_pred_flat_norm[0])

print("\nX_true_raw shape :", X_true_raw.shape)
print("X_pred_raw shape :", X_pred_raw.shape)

# Axe Vg vrai : canal 11
Vg_true = X_true_raw[:, 11]

# ------------------------------------------------------------
# 10. Comparaison des courbes par canal, sauf Vg
# ------------------------------------------------------------
channels_to_plot = list(range(N_CHANNELS - 1)) 

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.ravel()

for ax_idx, ch in enumerate(channels_to_plot):
    ax = axes[ax_idx]

    ax.plot(
        Vg_true,
        X_true_raw[:, ch],
        label="True X_test",
        linewidth=2.0
    )

    ax.plot(
        Vg_true,
        X_pred_raw[:, ch],
        "--",
        label="Pred surrogate(MLP params)",
        linewidth=2.0
    )

    ax.set_title(f"Canal {ch} : {channel_names[ch]}")
    ax.set_xlabel("Vg [V]")
    ax.set_ylabel(channel_names[ch])
    ax.grid(True)
    ax.legend()

# Cacher le dernier subplot inutilisé, car 11 canaux à tracer dans 12 cases
for k in range(len(channels_to_plot), len(axes)):
    axes[k].axis("off")

plt.suptitle(
    f"Comparaison courbes X_test vs surrogate(MLP params) — sample_index={sample_index}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ------------------------------------------------------------
# 11. Optionnel : sauvegarder les paramètres prédits
# ------------------------------------------------------------
out_txt = f"parameters_Xtest_sample_{sample_index}_mlp.txt"

with open(out_txt, "w", encoding="utf-8") as f:
    for name, value in zip(param_names, Y_pred[0]):
        f.write(f"+{name:>10s}_nhv = {value:.8e}\n")

print(f"\nParamètres prédits sauvegardés dans : {out_txt}")